# Analyze Lighthouse

Experimentation with analyzing Lighthouse scores.

In [2]:
import sys
import ast
from pathlib import Path

In [3]:
import pandas as pd
import altair as alt

In [4]:
this_dir = Path("__file__").parent.absolute()

In [5]:
sys.path.append(str(this_dir.parent / "newshomepages"))

In [6]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [7]:
analysis_dir = this_dir.parent / "_analysis"

Read in the dataframe

In [45]:
df = pd.read_csv(
    extracts_dir / "lighthouse-sample.csv",
    usecols=[
        'handle',
        'file_name',
        'date',
        'performance',
        'accessibility',
    ],
    dtype={
        'handle': str,
        'file_name': str,
        'performance': float,
        'accessibility': float,
    },
    parse_dates=["date"]
)

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11503 entries, 0 to 11502
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   handle         11503 non-null  object        
 1   file_name      11503 non-null  object        
 2   date           11503 non-null  datetime64[ns]
 3   performance    11501 non-null  float64       
 4   accessibility  11503 non-null  float64       
dtypes: datetime64[ns](1), float64(2), object(2)
memory usage: 449.5+ KB


Exclude any sites with less than 10 observations

In [47]:
observations_by_site = df.groupby("handle").size().rename("n").reset_index()

In [48]:
not_qualified = observations_by_site[observations_by_site.n < 10]

In [49]:
qualified_df = df[~df.handle.isin(not_qualified)]

Aggregate descriptive statistics for each metric.

In [64]:
agg_df = qualified_df.groupby("handle").agg({
    'performance': ['count', 'median', 'mean', 'min', 'max', 'std'],
    'accessibility': ['count', 'median', 'mean', 'min', 'max', 'std'],
})

In [65]:
agg_df

performance                                        accessibility  \
                   count median      mean   min   max       std         count   
handle                                                                          
100Reporters          13  0.160  0.163077  0.14  0.20  0.018879            13   
11AliveNews           13  0.220  0.220000  0.15  0.28  0.040208            13   
12NewsNow             13  0.170  0.219231  0.08  0.36  0.086068            13   
12khari               13  0.160  0.147692  0.04  0.31  0.069421            13   
13wmaznews            12  0.245  0.266667  0.14  0.38  0.081389            12   
...                  ...    ...       ...   ...   ...       ...           ...   
yorkdispatch          13  0.830  0.819231  0.70  0.96  0.066641            13   
zeitonline            13  0.510  0.500000  0.39  0.58  0.055976            13   
zerohedge             19  0.460  0.438947  0.26  0.54  0.075638            19   
zerohora              19  0.260  0.265263  0.24  0.30  0.016114            19   
zn_ua                 13  0.250  0.248462  0.21  0.30  0.028823            13   

                                                     
             median      mean   min   max       std  
handle                                               
100Reporters   0.89  0.890000  0.89  0.89  0.000000  
11AliveNews    0.81  0.819231  0.81  0.83  0.010377  
12NewsNow      0.81  0.813077  0.81  0.83  0.007511  
12khari        0.78  0.780000  0.78  0.78  0.000000  
13wmaznews     0.82  0.820000  0.81  0.83  0.010445  
...             ...       ...   ...   ...       ...  
yorkdispatch   0.88  0.892308  0.88  1.00  0.034194  
zeitonline     0.90  0.889231  0.86  0.90  0.013205  
zerohedge      0.95  0.950000  0.95  0.95  0.000000  
zerohora       0.84  0.840000  0.84  0.84  0.000000  
zn_ua          0.66  0.660000  0.66  0.66  0.000000  

[802 rows x 12 columns]

Flatten the dataframe

In [69]:
flat_df = agg_df.copy()
flat_df.columns = ['_'.join(col) for col in flat_df.columns]

Classify the scores

In [71]:
def color_code(val):
    """Return the classification of a metric according to Google's system.
    
    Source: https://developer.chrome.com/docs/lighthouse/performance/performance-scoring/
    """
    if val >= .9:
        return 'green'
    elif val >= .5:
        return 'orange'
    else:
        return 'red'

In [72]:
flat_df['performance_color'] = flat_df.performance_median.apply(color_code)

In [81]:
flat_df['accessibility_color'] = flat_df.accessibility_median.apply(color_code)

Rank the result

In [95]:
flat_df['performance_rank'] = flat_df.performance_median.rank(ascending=False, method="min")

In [96]:
flat_df.sort_values("performance_rank")[[
    'performance_median',
    'performance_rank'
]]

,performance_median,performance_rank
handle,,
tass_agency,1.00,1.0
insideclimate,1.00,1.0
YediotAhronot,0.98,3.0
techmeme,0.97,4.0
gigharbornow,0.96,5.0
...,...,...
JournalStarNews,0.02,798.0
BostonHerald,0.02,798.0
portalimprensa,0.02,798.0


Total up the colors

In [79]:
flat_df.performance_color.value_counts(normalize=True)

red       0.781796
orange    0.205736
green     0.012469
Name: performance_color, dtype: float64

In [82]:
flat_df.accessibility_color.value_counts(normalize=True)

orange    0.687032
green     0.306733
red       0.006234
Name: accessibility_color, dtype: float64

In [83]:
flat_df.performance_median.describe()

count    802.000000
mean       0.347126
std        0.215802
min        0.010000
25%        0.190000
50%        0.275000
75%        0.470000
max        1.000000
Name: performance_median, dtype: float64

In [84]:
flat_df.accessibility_median.describe()

count    802.000000
mean       0.843691
std        0.100849
min        0.430000
25%        0.780000
50%        0.860000
75%        0.920000
max        1.000000
Name: accessibility_median, dtype: float64